# B6.4 — Baseline II: OmniCount (official code, re-run by us)

Pre-declared in `results/B6_0_Setup_and_Rules/B6_rule_file.json`: *OmniCount — official code, time-boxed; reproduction check FSC-147 MAE 21.46*; if it does not reproduce, *report our re-run and the published figure, gap explained*.

| Dataset | Queries per image (rows key) | Protocol in B6_5 |
|---|---|---|
| FSC-147 test (1,190) | the class (`OmniCount`) | single-class MAE (+ RMSE, NAE, SRE) |
| CARPK test (459) | "car" (`OmniCount`) | single-class |
| VOC 2007 test (4,952) | present classes (`OmniCount`); on the 500-image subset also all 20 classes (`OmniCount_std`) | present classes (full test); standard (subset) |
| OmniCount-191 test (1,957) | present classes (`OmniCount`); on Fruits (303) also the domain vocabulary (`OmniCount_std`) | present classes (full test); standard (Fruits) |

**What is run.** The authors' code at commit `931b2a4` with its pinned submodules (SAN `e19262d`, Marigold `ac91528`, CLIP-Surgery `d4696d4`, SAM `dca509f`) and its vendored TFOC SAM fork. The repo has no end-to-end runner; its stage scripts are ported here **following the code where it differs from the paper**:
SAN ViT-L/14 per class (COCO classes merged into the vocabulary) → Marigold v1.0 depth (10 steps, ensemble 10, 768 px) → refinement (Otsu; 10×10 window, not in another query class's mask, |depth − class mean| < 0.2; radius-5 opening; empty SAN mask → depth < 0.5) → reference points = local maxima of (1 − mean CLIP-Surgery similarity over a fixed 59-class list) inside the refined mask → TFOC point-prompted SAM ViT-B on the masked patch (vanilla SAM everything mode on the patch as the authors' fallback) → count.

**Reconstructed (missing from the repo):** the SAN binary-mask writer — SAN `predict.py`, one class per call, mask = pixels whose argmax is the query class.
**Behaviour-neutral engineering changes (tested):** models loaded once; unused CLIPSeg call removed; pixel-loop refinement vectorised (identical output — re-checked on real images in Step 1.1); CLIP text features of the fixed list computed once; Marigold seed fixed to 0 per image (the wrapper seeds from the clock); compatibility patches printed in Step 0.2.
**Fixed before any test count** (saved in `B6_4_declarations.json`, Step 0.5): per-dataset settings — FSC-147 `core/sam_mod.py`, CARPK `core/sam_mod_aerial.py`, VOC and OmniCount-191 `core/sam_mod_multi.py`; **count = number of masks** (primary, the paper's description); the active rule in `visualization/count.py` (masks with above-average overlay intensity) is stored as `*_countpy` (secondary).

**Two sessions in parallel:** `B6_4_Baseline_OmniCount_A.ipynb` and `_B.ipynb` are identical except `PART` (Step 0.2). They share the work through the per-image rows on Drive (each re-checks a row before computing it): FSC-147 together (A from the start, B from the end) → both evaluate the gate → A: VOC → OmniCount-191 → CARPK; B: CARPK → OmniCount-191 → VOC (reversed). Start both at about the same time, each on an **L4** runtime. Near a meeting point a few images may be computed twice (Drive sync lag) — identical results.

**Order:** sanity check on 3 FSC-147 *val* images (port = authors' refinement on real masks; timing) → FSC-147 test + **reproduction gate** (stops if MAE is outside 21.46 ± 10%, i.e. 19.31–23.61) → CARPK, VOC, OmniCount-191.
**Runtime (L4):** about 100 GPU-hours in total (Step 1.1/1.2 on val: depth ≈ 32 s per image dominates) → about 50 h wall-clock with two sessions; FSC-147 + gate ≈ 6 h. **Resumable** — Run all again to continue.
**Speed check (29 Oct, 3 FSC-147 val images, exploratory):** Marigold batch size gives no speed gain on L4 (auto = 2); fp16 is 1.7× faster but changes depth by ≈1.5× the seed-to-seed noise → the declared fp32 setting is kept.

**Before leaving it overnight:** Step 0.2 shows the commits and patch diff; Step 1.1 prints `REFINEMENT EQUIVALENCE: PASS` and the time estimate; Step 2.1 prints `REPRODUCTION GATE: PASS` (or stops).

In [1]:
# Step 0.1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 0.2 — Paths, installs (B6_1 stack), our repo, OmniCount at a fixed commit + pinned submodules, documented patches, weights
# ============================================================
import hashlib, json, shutil, subprocess, sys, tarfile, time, urllib.request, zipfile
from datetime import datetime, timezone
from pathlib import Path
import numpy as np, pandas as pd
import torch
PART = "A"                    # "A" in B6_4_Baseline_OmniCount_A.ipynb, "B" in ..._B.ipynb — the only difference between the two
ALLOW_NON_L4 = False          # both sessions on L4 (T4 vs L4 can shift counts by 0–3)
assert PART in ("A", "B")
assert torch.cuda.is_available(), "Use a GPU runtime (L4)."
GPU_NAME = torch.cuda.get_device_name(0); print("GPU:", GPU_NAME, "| PART", PART)
assert "L4" in GPU_NAME or ALLOW_NON_L4, f"{GPU_NAME}: use an L4 runtime (or set ALLOW_NON_L4 = True deliberately)"
PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"; DATASETS = PROJECT_ROOT / "datasets"
OUT_DIR = RESULTS_ROOT / "B6_4_Baseline_OmniCount"; OUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/data"); LOCAL.mkdir(parents=True, exist_ok=True)

# --- Same package stack as B6_1/B6_2 (SAN + detectron2 + diffusers, protobuf last) + TFOC/CLIP-Surgery extras
DETECTRON2_COMMIT = "1e3e13bbf607b54f62205c4c33922521822fb298"
!pip install -q cython scipy shapely timm h5py submitit scikit-image wandb setuptools fvcore tabulate regex opencv-python "open_clip_torch==2.16.0" ftfy selective-search
!pip uninstall -y -q detectron2
!python -m pip install -q --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git@{DETECTRON2_COMMIT}"
!pip install -q "accelerate==1.14.0" "diffusers==0.40.0" "transformers==5.16.1"
!pip install -q "protobuf==3.20.3" "iopath==0.1.10"

REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
subprocess.run(["rm", "-rf", str(REPO_DIR)])
subprocess.run(["git", "clone", "-q", "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git", str(REPO_DIR)], check=True)
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

# --- OmniCount official code at a fixed commit; submodules at the commits pinned by that commit (GroundingDINO not needed)
OMNI_COMMIT = "931b2a4"
SUBMODULES = {"external/SAN": "e19262d", "external/Marigold": "ac91528", "external/clips": "d4696d4", "external/sam": "dca509f"}
OMNI_DIR = Path("/content/OmniCount"); OMNI_EXT = OMNI_DIR / "external"
PATCHES = [  # (file, old, new) — compatibility only, no change to the method
    ("external/tfoc/utils.py", "from scipy.ndimage.filters import gaussian_filter",
     "from scipy.ndimage import gaussian_filter  # patched: scipy.ndimage.filters removed"),
    ("external/tfoc/clip/clip.py", "from pkg_resources import packaging", "import packaging.version  # patched: pkg_resources deprecated"),
    ("external/clips/clip/clip.py", "from pkg_resources import packaging", "import packaging.version  # patched: pkg_resources deprecated"),
]
if not (OMNI_EXT / "SAN" / "predict.py").exists():
    subprocess.run(["rm", "-rf", str(OMNI_DIR)])
    subprocess.run(["git", "clone", "-q", "https://github.com/mondalanindya/OmniCount.git", str(OMNI_DIR)], check=True)
    subprocess.run(["git", "-C", str(OMNI_DIR), "checkout", "-q", OMNI_COMMIT], check=True)
    subprocess.run(["git", "-C", str(OMNI_DIR), "submodule", "update", "--init", "-q", *SUBMODULES], check=True)
    for f, old, new in PATCHES:
        p = OMNI_DIR / f; s = p.read_text(); assert old in s, (f, old); p.write_text(s.replace(old, new))
OMNI_HEAD = subprocess.check_output(["git", "-C", str(OMNI_DIR), "rev-parse", "HEAD"], text=True).strip()
assert OMNI_HEAD.startswith(OMNI_COMMIT), OMNI_HEAD
SUB_HEADS = {}
for sub, c in SUBMODULES.items():
    SUB_HEADS[sub] = subprocess.check_output(["git", "-C", str(OMNI_DIR / sub), "rev-parse", "HEAD"], text=True).strip()
    assert SUB_HEADS[sub].startswith(c), (sub, SUB_HEADS[sub])
print(subprocess.run(["git", "-C", str(OMNI_DIR), "diff", "--", "external/tfoc"], capture_output=True, text=True).stdout)
print(subprocess.run(["git", "-C", str(OMNI_EXT / "clips"), "diff"], capture_output=True, text=True).stdout)

# --- Weights (cached on Drive)
CKPT = PROJECT_ROOT / "Checkpoints"
SAM_B = CKPT / "SAM" / "sam_vit_b_01ec64.pth"; SAM_B.parent.mkdir(parents=True, exist_ok=True)
if not SAM_B.exists():
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", SAM_B)
h = hashlib.md5(SAM_B.read_bytes()).hexdigest(); assert h.startswith("01ec64"), h   # SAM checkpoint names carry the MD5 prefix
SAN_L = CKPT / "SAN" / "san_vit_large_14.pth"; SAN_L.parent.mkdir(parents=True, exist_ok=True)
if not SAN_L.exists():                                             # official SAN release (predict.py: Mendel192/san)
    from huggingface_hub import hf_hub_download
    shutil.copy2(hf_hub_download("Mendel192/san", filename="san_vit_large_14.pth"), SAN_L)
SAN_L_SHA256 = hashlib.sha256(SAN_L.read_bytes()).hexdigest()
CLIP_SHA = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"   # ViT-B/16 = CS-ViT-B/16 weights (as B6_3)
CLIP_CACHE = Path.home() / ".cache" / "clip"; CLIP_CACHE.mkdir(parents=True, exist_ok=True)
CLIP_FILE = CLIP_CACHE / "ViT-B-16.pt"; CLIP_DRIVE = CKPT / "CLIP" / "ViT-B-16.pt"
if CLIP_DRIVE.exists() and not CLIP_FILE.exists():
    shutil.copy2(CLIP_DRIVE, CLIP_FILE)
if not CLIP_FILE.exists() or hashlib.sha256(CLIP_FILE.read_bytes()).hexdigest() != CLIP_SHA:
    for host in ["openaipublic.azureedge.net", "openaipublic.blob.core.windows.net"]:
        try:
            urllib.request.urlretrieve(f"https://{host}/clip/models/{CLIP_SHA}/ViT-B-16.pt", CLIP_FILE)
            if hashlib.sha256(CLIP_FILE.read_bytes()).hexdigest() == CLIP_SHA:
                break
        except Exception as e:
            print("CLIP download failed from", host, "-", type(e).__name__)
assert hashlib.sha256(CLIP_FILE.read_bytes()).hexdigest() == CLIP_SHA, "CLIP weights not available"
CLIP_DRIVE.parent.mkdir(parents=True, exist_ok=True)
if not CLIP_DRIVE.exists():
    shutil.copy2(CLIP_FILE, CLIP_DRIVE)
print("Repo:", REPO_COMMIT[:10], "| OmniCount:", OMNI_HEAD[:7], "|", {k.split("/")[-1]: v[:7] for k, v in SUB_HEADS.items()},
      "| SAM ViT-B, SAN ViT-L/14 (sha256", SAN_L_SHA256[:12] + "), CLIP ViT-B/16 verified")

GPU: NVIDIA L4 | PART A
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpc-google-iam-v1 0.14.6 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-dataproc 5.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.85.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 3.20.3 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have

In [3]:
# ============================================================
# Step 0.3 — Load models: SAN ViT-L/14, Marigold v1.0, CLIP-Surgery CS-ViT-B/16, SAM ViT-B (TFOC fork) + vanilla AMG
# ============================================================
import contextlib, importlib.util, logging, os
import google.protobuf, importlib.metadata as md_meta
assert md_meta.version("protobuf") == "3.20.3" and google.protobuf.__version__ == "3.20.3", \
    "protobuf mismatch — restart the session and Run all again."
from tqdm.auto import tqdm
sys.path.insert(0, str(REPO_DIR))
import src.datasets as D                                            # ours first: Marigold's repo also has a top-level `src`
for p in [str(OMNI_EXT / "SAN"), str(OMNI_EXT / "Marigold"), str(OMNI_EXT), str(OMNI_EXT / "sam")]:   # sam: = their `pip install -e external/sam`
    if p not in sys.path:
        sys.path.append(p)                                          # appended so our `src` keeps priority

def _load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path); mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod); return mod

@contextlib.contextmanager
def _cwd(p):
    old = os.getcwd(); os.chdir(p)
    try:
        yield
    finally:
        os.chdir(old)

# --- SAN ViT-L/14 through SAN's own predict.py (commit e19262d)
with _cwd(OMNI_EXT / "SAN"):
    san_predict = _load_module("san_predict", OMNI_EXT / "SAN" / "predict.py")
    san_predictor = san_predict.Predictor(config_file=str(OMNI_EXT / "SAN" / "configs" / "san_clip_vit_large_res4_coco.yaml"),
                                          model_path=str(SAN_L))
# --- Marigold (authors' wrapper: MarigoldPipeline, fp32 — no --half_precision); seed_all moved to src/util/seeding.py
from marigold import MarigoldPipeline
marigold_seeding = _load_module("marigold_seeding", OMNI_EXT / "Marigold" / "src" / "util" / "seeding.py")
MARIGOLD_CKPT, marigold_pipe = None, None
for ck in ["Bingxin/Marigold", "prs-eth/marigold-depth-v1-0"]:     # wrapper default first; same v1.0 weights as fallback
    try:
        marigold_pipe = MarigoldPipeline.from_pretrained(ck); MARIGOLD_CKPT = ck; break
    except Exception as e:
        print("Marigold checkpoint", ck, "failed:", type(e).__name__, str(e)[:150])
assert marigold_pipe is not None, "No Marigold v1.0 checkpoint could be loaded"
marigold_pipe = marigold_pipe.to("cuda")
# --- CLIP-Surgery (the clips submodule, as imported by core/sam_mod*.py)
from clips import clip as cs_clip
assert Path(cs_clip.__file__).resolve().is_relative_to(OMNI_EXT / "clips"), cs_clip.__file__
cs_model, _ = cs_clip.load("CS-ViT-B/16", device="cuda"); cs_model.eval()
# --- SAM ViT-B from the TFOC fork (point-prompt AMG) + the official AMG for the fallback, as in core/sam_mod*.py
from tfoc.shi_segment_anything import sam_model_registry as tfoc_registry
from tfoc.shi_segment_anything.automatic_mask_generator import SamAutomaticMaskGenerator as TfocAMG
from sam.segment_anything.automatic_mask_generator import SamAutomaticMaskGenerator as VanillaAMG
assert Path(sys.modules[TfocAMG.__module__].__file__).resolve().is_relative_to(OMNI_EXT / "tfoc")
assert Path(sys.modules[VanillaAMG.__module__].__file__).resolve().is_relative_to(OMNI_EXT / "sam")
import segment_anything as _sa; assert Path(_sa.__file__).resolve().is_relative_to(OMNI_EXT / "sam"), _sa.__file__   # pinned dca509f, not a pip copy
sam_b = tfoc_registry["vit_b"](checkpoint=str(SAM_B)).to("cuda")
print("Models loaded: SAN ViT-L/14 | Marigold", MARIGOLD_CKPT, "| CLIP-Surgery CS-ViT-B/16 | SAM ViT-B (TFOC fork) + vanilla AMG")

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
100%|████████████████████████████████████████| 934M/934M [00:06<00:00, 143MiB/s]
/content/OmniCount/external/SAN/san/model/side_adapter/timm_wrapper.py:24: UserWarning: Unused kwargs are provided:{'dynamic_img_pad': False, 'device': None, 'dtype': None}.
  warnings.warn(f"Unused kwargs are provided:{kwargs}.")


Loading model from:  /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/Checkpoints/SAN/san_vit_large_14.pth
Loaded model from:  /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/Checkpoints/SAN/san_vit_large_14.pth


model_index.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

The config attributes {'prediction_type': 'depth'} were passed to MarigoldDepthPipeline, but are not expected and will be ignored. Please verify your model_index.json configuration file.
Keyword arguments {'prediction_type': 'depth'} are not expected by MarigoldDepthPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

Models loaded: SAN ViT-L/14 | Marigold Bingxin/Marigold | CLIP-Surgery CS-ViT-B/16 | SAM ViT-B (TFOC fork) + vanilla AMG


In [4]:
# ============================================================
# Step 0.4 — Test items (as B6_3); OmniCount's counts depend on the query set, so the subsets get a second set (query_std)
# ============================================================
RULE = json.load(open(RESULTS_ROOT / "B6_0_Setup_and_Rules" / "B6_rule_file.json"))
SUBSETS = json.load(open(RESULTS_ROOT / "B6_0_Setup_and_Rules" / "B6_0_baseline_subsets.json"))
VOC_SUB, FRUITS_SUB = set(SUBSETS["voc_standard_500"]), set(SUBSETS["omnicount_fruits_test"])
ITEMS = {"fsc147": [], "carpk": [], "voc": [], "omnicount": []}

fsc = D.FSC147Dataset(DATASETS / "FSC147" / "raw", split="test", load_images=False)
for i in range(len(fsc)):
    s = fsc[i]; c = next(iter(s["categories"]))
    ITEMS["fsc147"].append({"key": f"{i:05d}", "image_path": s["image_path"], "query": [c], "query_std": None, "prompts": {c: c},
                            "meta": {"dataset": "fsc147", "image_id": s["image_id"], "gt": {c: s["categories"][c]["count"]}}})
CARPK_LOCAL = LOCAL / "carpk"
if not any(CARPK_LOCAL.rglob("test_annotations.json")):
    with zipfile.ZipFile(DATASETS / "CARPK_kaggle_coco.zip") as z:
        z.extractall(CARPK_LOCAL)
cds = D.CARPKDataset(CARPK_LOCAL, "test", load_images=False)
for i in range(len(cds)):
    s = cds[i]
    ITEMS["carpk"].append({"key": Path(s["image_path"]).stem, "image_path": s["image_path"], "query": ["car"], "query_std": None,
                           "prompts": {"car": "car"},
                           "meta": {"dataset": "carpk", "image_id": s["image_id"], "gt": {"car": s["categories"]["car"]["count"]}}})
VOC_LOCAL = LOCAL / "voc2007"
if not (VOC_LOCAL / "VOCdevkit" / "VOC2007" / "ImageSets").exists():
    with tarfile.open(DATASETS / "VOCtest_06-Nov-2007.tar") as t:
        t.extractall(VOC_LOCAL, filter="data")
voc = D.VOC2007CountingDataset(VOC_LOCAL, "test", include_difficult=True, load_images=False)
for i in range(len(voc)):
    s = voc[i]; vid = Path(s["image_path"]).stem
    present = list(s["categories"])                                         # >= 1 annotated object (primary convention)
    std = list(D.VOC_CLASSES) if vid in VOC_SUB else None
    ITEMS["voc"].append({"key": vid, "image_path": s["image_path"], "query": present, "query_std": std,
                         "prompts": {c: D.voc_prompt_name(c) for c in (std or present)},
                         "meta": {"dataset": "voc2007", "domain": "VOC2007", "image_id": s["image_id"], "in_standard_subset": vid in VOC_SUB,
                                  "gt_all": {c: v["count_all"] for c, v in s["categories"].items()},
                                  "gt_nondifficult": {c: v["count_nondifficult"] for c, v in s["categories"].items() if v["count_nondifficult"] > 0}}})
om = D.OmniCount191Dataset(DATASETS / "OmniCount-191", split="test", load_images=False)
for i in range(len(om)):
    s = om[i]; d = s["domain"]; present = list(s["categories"])
    std = list(om.domain_vocabulary[d]) if s["image_id"] in FRUITS_SUB else None
    ITEMS["omnicount"].append({"key": s["image_id"].replace("/", "__"), "image_path": s["image_path"], "query": present, "query_std": std,
                               "prompts": {c: D.omnicount_prompt_name(c) for c in dict.fromkeys(present + (std or []))},
                               "meta": {"dataset": "omnicount", "domain": d, "image_id": s["image_id"], "in_standard_subset": s["image_id"] in FRUITS_SUB,
                                        "vocabulary": list(om.domain_vocabulary[d]), "gt": {c: v["count"] for c, v in s["categories"].items()}}})
assert len(ITEMS["fsc147"]) == 1190 and len(ITEMS["carpk"]) == 459 and len(ITEMS["voc"]) == 4952 and len(ITEMS["omnicount"]) == 1957
assert sum(it["meta"]["in_standard_subset"] for it in ITEMS["voc"]) == 500 and sum(it["meta"]["in_standard_subset"] for it in ITEMS["omnicount"]) == len(FRUITS_SUB)
assert all(set(it["query"]) <= set(it["query_std"]) for k in ("voc", "omnicount") for it in ITEMS[k] if it["query_std"])
for k, v in ITEMS.items():
    n_img = len(v); n_q = sum(len(it["query"]) for it in v); n_std = sum(len(it["query_std"] or []) for it in v)
    n_san = sum(len(set(it["query"]) | set(it["query_std"] or [])) for it in v)
    print(f"{k}: {n_img} images | present-class queries {n_q} | standard-subset queries {n_std} | SAN masks {n_san}")

fsc147: 1190 images | present-class queries 1190 | standard-subset queries 0 | SAN masks 1190
carpk: 459 images | present-class queries 459 | standard-subset queries 0 | SAN masks 459
voc: 4952 images | present-class queries 7632 | standard-subset queries 10000 | SAN masks 16868
omnicount: 1957 images | present-class queries 3754 | standard-subset queries 2424 | SAN masks 4952


In [5]:
# ============================================================
# Step 0.5 — OmniCount port: geometric refinement + reference points / SAM counting
# ============================================================
# ---- OmniCount port, part 1: geometry-aware object recovery (preprocessing/extract_bin_mask.py + extract_patch.py)
import numpy as np, cv2
from scipy import ndimage

REFINE_TOL, REFINE_WIN, OPEN_RADIUS = 0.2, 5, 5                   # tolerance = 0.2, win = 5, radius = 5 (both scripts)
STREL_DIAMOND = ndimage.iterate_structure(ndimage.generate_binary_structure(2, 1), OPEN_RADIUS)   # extract_bin_mask.py

def _create_disk(r):                                               # extract_patch.py create_disk
    x = np.arange(-r, r + 1); y = np.arange(-r, r + 1); y, x = np.meshgrid(x, x)
    return x * x + y * y <= r * r
STREL_DISK = _create_disk(OPEN_RADIUS)

def normalise_depth(depth_u16):
    """Both scripts: depth read from the uint16 PNG, then min-max normalised."""
    d = depth_u16
    return (d - d.min()) / (d.max() - d.min())

def _window_any(mask_bool, win=REFINE_WIN):
    """near[k, j] = any mask pixel (x, y) with k in range(x-win, x+win) and j in range(y-win, y+win),
    i.e. x in [k-win+1, k+win], y in [j-win+1, j+win] (the original loop's asymmetric 2*win window)."""
    H, W = mask_bool.shape
    pad = np.pad(mask_bool.astype(np.int32), ((win, win), (win, win)))
    S = np.zeros((H + 2 * win + 1, W + 2 * win + 1), np.int64)
    S[1:, 1:] = pad.cumsum(0).cumsum(1)
    # window rows x in [k-win+1, k+win] -> padded indices [k+1, k+2*win]; same for columns
    k0, k1 = np.arange(H) + 1, np.arange(H) + 2 * win + 1
    j0, j1 = np.arange(W) + 1, np.arange(W) + 2 * win + 1
    tot = S[np.ix_(k1, j1)] - S[np.ix_(k0, j1)] - S[np.ix_(k1, j0)] + S[np.ix_(k0, j0)]
    return tot > 0

def _refined_before_opening(binary_mask_otsu, other_masks, depth_norm):
    """Vectorised process_binary_mask (before the opening) — identical output to the original pixel loops."""
    sel = binary_mask_otsu == 255
    if not sel.any():                                              # is_valid_sem == 0 -> depth < 0.5 fallback
        return depth_norm < 0.5
    mean_depth = np.mean(depth_norm[np.nonzero(binary_mask_otsu)])
    other_any = np.zeros_like(sel)
    for om in other_masks:
        other_any |= (om == 255)
    return _window_any(sel) & ~other_any & (np.abs(depth_norm - mean_depth) < REFINE_TOL)

def refine_class(san_mask_u8, other_masks_u8, depth_norm):
    """Returns (refined_bin_mask uint8 0/255 [extract_bin_mask.py, diamond], patch_mask bool [extract_patch.py, disk], san_empty)."""
    _, otsu = cv2.threshold(san_mask_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    pre = _refined_before_opening(otsu, other_masks_u8, depth_norm)
    refined_bin = ndimage.binary_opening(pre, structure=STREL_DIAMOND).astype(np.uint8) * 255
    patch_mask = ndimage.binary_opening(pre, structure=STREL_DISK)
    return refined_bin, patch_mask, not (otsu == 255).any()

def refine_image(san_masks, depth_u16, rgb):
    """san_masks: {class: uint8 0/255} for the QUERY SET of this run (other masks = the rest of the query set).
    rgb: HxWx3 uint8 (RGB). Returns {class: (patch_rgb uint8, refined_bin uint8, san_empty)}."""
    depth_norm = normalise_depth(depth_u16)
    if depth_norm.shape[:2] != rgb.shape[:2]:                      # extract_patch.py resize (INTER_NEAREST)
        depth_norm = cv2.resize(depth_norm, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST)
    out = {}
    for c, m in san_masks.items():
        if m.shape[:2] != rgb.shape[:2]:
            m = cv2.resize(m, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST)
        others = [o if o.shape[:2] == rgb.shape[:2] else cv2.resize(o, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST)
                  for k, o in san_masks.items() if k != c]
        refined_bin, patch_mask, empty = refine_class(m, others, depth_norm)
        patch = rgb.copy(); patch[~patch_mask] = 0                 # extract_patch: area outside the mask set to black
        out[c] = (patch, refined_bin, empty)
    return out

# ---- OmniCount port, part 2: reference points (CLIP-Surgery) + TFOC point-prompt SAM counting (core/sam_mod*.py)
import numpy as np, cv2, torch
from PIL import Image
from skimage.feature import peak_local_max
from torchvision.transforms import Compose, Resize, ToTensor, Normalize, InterpolationMode

# Per-dataset settings, copied from the authors' scripts (fixed before any test count; see the header cell)
#   min_points: fall back to vanilla SAM everything mode on the patch if fewer reference points
#   min_masks : fall back if the point-prompted generator returns fewer masks (0 = never; their "< 0" check)
OMNI_CFG = {
    "fsc147":    {"script": "core/sam_mod.py",        "min_distance": 4,  "threshold_rel": 0.8, "min_points": 5, "min_masks": 5},
    "carpk":     {"script": "core/sam_mod_aerial.py", "min_distance": 6,  "threshold_rel": 0.6, "min_points": 1, "min_masks": 0},
    "voc":       {"script": "core/sam_mod_multi.py",  "min_distance": 10, "threshold_rel": 0.9, "min_points": 1, "min_masks": 0},
    "omnicount": {"script": "core/sam_mod_multi.py",  "min_distance": 10, "threshold_rel": 0.9, "min_points": 1, "min_masks": 0},
}
# Fixed 59-class list used by every sam_mod*.py for the CLIP-Surgery similarity (verbatim)
ALL_TEXTS = ['airplane', 'bag', 'strawberries', 'bedclothes', 'bench', 'bicycle', 'bird', 'boat', 'book', 'bottle', 'building',
             'bus', 'cabinet', 'car', 'cat', 'ceiling', 'chair', 'cloth', 'computer', 'cow', 'cup', 'curtain', 'dog', 'door', 'fence',
             'floor', 'flower', 'food', 'grass', 'ground', 'horse', 'keyboard', 'light', 'motorbike', 'mountain', 'mouse', 'person',
             'plate', 'platform', 'potted plant', 'road', 'rock', 'sheep', 'shelves', 'sidewalk', 'sign', 'sky', 'snow', 'sofa',
             'table', 'track', 'train', 'tree', 'truck', 'tv monitor', 'wall', 'water', 'window', 'wood']
PREPROCESS_512 = Compose([Resize((512, 512), interpolation=InterpolationMode.BICUBIC), ToTensor(),
                          Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))])

class OmniCounter:
    """refine_count() of core/sam_mod*.py with the models loaded once (the original reloads CLIP and SAM per call).
    The CLIPSeg block of the original is omitted: its output (`result`) is never used."""
    def __init__(self, clip_mod, cs_model, sam_model, TfocAMG, VanillaAMG, device="cuda"):
        self.clip, self.cs, self.sam, self.TfocAMG, self.VanillaAMG, self.device = clip_mod, cs_model, sam_model, TfocAMG, VanillaAMG, device
        with torch.no_grad():                                       # constant across images (fixed list) -> computed once
            self.text_features = clip_mod.encode_text_with_prompt_ensemble(cs_model, ALL_TEXTS, device)

    def reference_points(self, pil_img, pred, cfg):
        image = PREPROCESS_512(pil_img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            image_features = self.cs.encode_image(image)
            image_features = image_features / image_features.norm(dim=1, keepdim=True)
            similarity = self.clip.clip_feature_surgery(image_features, self.text_features)
            similarity_map = self.clip.get_similarity_map(similarity[:, 1:, :], pred.shape[:2])
        new_mask = similarity_map.mean(dim=3).squeeze(0).cpu().detach().numpy()
        exp_mask1 = torch.from_numpy(pred).float().unsqueeze(0).numpy() / 255.0
        hadamard = ((1 - new_mask) * exp_mask1).squeeze()
        smoothed = cv2.GaussianBlur(hadamard, (5, 5), sigmaX=0)
        coordinates = peak_local_max(smoothed, min_distance=cfg["min_distance"], threshold_rel=cfg["threshold_rel"])
        return [[y, x] for x, y in coordinates]                     # (row, col) -> (x, y), as in the original

    def count(self, patch_rgb, refined_bin, cfg):
        pil_img = Image.fromarray(patch_rgb)                        # the original re-reads the saved PNG patch (lossless)
        img = np.asarray(pil_img)
        pts = self.reference_points(pil_img, refined_bin, cfg)
        fallback = "none"
        if len(pts) == 0 or len(pts) < cfg["min_points"]:
            masks = self.VanillaAMG(self.sam).generate(img); fallback = "few_points"
        else:
            masks = self.TfocAMG(self.sam).generate(img, pts)
            if len(masks) < cfg["min_masks"]:
                masks = self.VanillaAMG(self.sam).generate(img); fallback = "few_masks"
        segs = [m["segmentation"] if m["segmentation"].ndim == 2 else m["segmentation"][:, :, 0] for m in masks]
        # count.py (active block): keep masks whose overlay mean pixel value exceeds the class average
        ov = [float(np.mean(np.where(s[..., None], img, np.zeros_like(img)))) for s in segs]
        avg = sum(ov) / len(ov) if ov else 0.0
        n_cpy = sum(v > 1 * avg for v in ov)
        diag = {"n_points": len(pts), "fallback": fallback, "n_masks": len(segs), "n_countpy": int(n_cpy),
                "mask_area": [int(s.sum()) for s in segs], "overlay_mean": [round(v, 4) for v in ov]}
        return len(segs), int(n_cpy), diag

counter = OmniCounter(cs_clip, cs_model, sam_b, TfocAMG, VanillaAMG, device="cuda")

In [6]:
# ============================================================
# Step 0.6 — Per-image priors (Marigold, SAN), resumable runner shared by two sessions, helpers, pre-declared decisions
# ============================================================
# ---- OmniCount port, part 3: per-image priors (Marigold depth, SAN per-class masks) + resumable runner
import contextlib, io, json, logging, time
from pathlib import Path
import numpy as np, cv2
from PIL import Image

MARIGOLD_ARGS = {"denoising_steps": 10, "ensemble_size": 10, "processing_res": 768,    # marigold_run(_mod).py defaults
                 "match_input_res": True, "batch_size": 0}
MARIGOLD_SEED = 0          # the wrapper seeds once from the clock; fixed per image here (reproducible, resumable)

@contextlib.contextmanager
def _quiet():
    """Silence per-call prints (SAN prints the vocabulary) and Marigold's scheduler warnings."""
    root = logging.getLogger(); lvl = root.level; root.setLevel(logging.ERROR)
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            yield
    finally:
        root.setLevel(lvl)

def depth_u16(pipe, image_path, seed_all):
    """marigold_run_mod.py: depth_np -> uint16 PNG (depth * 65535); the refinement re-reads and min-max normalises it."""
    seed_all(MARIGOLD_SEED)
    with _quiet():
        out = pipe(Image.open(image_path), color_map=None, show_progress_bar=False, **MARIGOLD_ARGS)
    return (out.depth_np * 65535.0).astype(np.uint16)

def san_mask(predictor, image_path, prompt):
    """Reconstructed binary-mask writer (theirs is not in the repo): SAN predict.py at e19262d, one class per call,
    vocabulary merged with the COCO classes (their wrappers never pass --aug-vocab, so predict() merges COCO-all);
    mask = pixels whose argmax is the query class (index 0)."""
    with _quiet():
        r = predictor.predict(Image.open(image_path), [prompt], False)
    return (r["sem_seg"] == 0).astype(np.uint8) * 255

def stage_a(it, classes, models, cache_path=None):
    """Depth + SAN masks for the union of classes needed by this image's runs; optional npz cache (lossless)."""
    t0 = time.time(); depth, san = None, {}
    if cache_path is not None and Path(cache_path).exists():
        z = np.load(cache_path, allow_pickle=False)
        depth = z["depth"]; shape = tuple(z["shape"])
        for c, packed in zip(z["classes"].tolist(), z["san"]):
            san[c] = (np.unpackbits(packed)[: shape[0] * shape[1]].reshape(shape) * 255).astype(np.uint8)
    t_depth = t_san = 0.0
    if depth is None:
        t1 = time.time(); depth = depth_u16(models["marigold"], it["image_path"], models["seed_all"]); t_depth = time.time() - t1
    missing = [c for c in classes if c not in san]
    t1 = time.time()
    for c in missing:
        san[c] = san_mask(models["san"], it["image_path"], it["prompts"][c])
    t_san = time.time() - t1
    if cache_path is not None and (missing or t_depth > 0):
        cs = sorted(san); shape = depth.shape
        packed = np.stack([np.packbits(san[c] > 0) for c in cs]) if cs else np.zeros((0, 0), np.uint8)
        tmp = Path(str(cache_path) + f".{models.get('worker', 'A')}.tmp.npz")   # per-worker tmp (two sessions share Drive)
        np.savez_compressed(tmp, depth=depth, shape=np.array(shape), classes=np.array(cs), san=packed)
        tmp.replace(cache_path)
    return depth, san, {"t_depth_s": t_depth, "t_san_s": t_san, "t_stage_a_s": time.time() - t0}

RUNS = [("OmniCount", "query"), ("OmniCount_std", "query_std")]   # _std: all classes, fixed subsets only

def run_omnicount(items, rows_dir, dataset, models, counter, refine_image, cache_dir=None, reverse=False):
    cfg = models["cfg"][dataset]; worker = models.get("worker", "A")
    rows_dir = Path(rows_dir); rows_dir.mkdir(parents=True, exist_ok=True)
    if cache_dir is not None:
        Path(cache_dir).mkdir(parents=True, exist_ok=True)
    order = list(reversed(items)) if reverse else list(items)       # worker B walks the list from the end
    todo = [it for it in order if not (rows_dir / f"{it['key']}.json").exists()]
    n_done = 0
    for it in models["tqdm"](todo, desc=f"OmniCount {dataset} [{worker}]"):
        if (rows_dir / f"{it['key']}.json").exists():                # done meanwhile by the other session
            continue
        t0 = time.time(); counts, diag_runs, errors = {}, {}, {}; n_done += 1
        needed = list(dict.fromkeys(it["query"] + (it.get("query_std") or [])))
        try:
            depth, san, ta = stage_a(it, needed, models, None if cache_dir is None else Path(cache_dir) / f"{it['key']}.npz")
            bgr = cv2.imread(str(it["image_path"])); rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        except Exception as e:                                       # image-level failure: every query counted as 0
            msg = f"stage_a {type(e).__name__}: {str(e)[:200]}"; ta = {}
            for run, qkey in RUNS:
                if it.get(qkey):
                    counts[run] = {c: 0 for c in it[qkey]}; counts[run + "_countpy"] = {c: 0 for c in it[qkey]}
                    errors[run] = {c: msg for c in it[qkey]}
            san = None
        if san is not None:
            for run, qkey in RUNS:
                q = it.get(qkey)
                if not q:
                    continue
                counts[run], counts[run + "_countpy"], diag_runs[run], errors_run = {}, {}, {}, {}
                try:
                    ref = refine_image({c: san[c] for c in q}, depth, rgb)  # other masks = the rest of THIS run's query set
                except Exception as e:
                    ref = None; msg = f"refine {type(e).__name__}: {str(e)[:200]}"
                for c in q:
                    try:
                        if ref is None:
                            raise RuntimeError(msg)
                        patch, refined_bin, empty = ref[c]
                        n, n_cpy, d = counter.count(patch, refined_bin, cfg); d["san_empty"] = bool(empty)
                        counts[run][c], counts[run + "_countpy"][c], diag_runs[run][c] = n, n_cpy, d
                    except Exception as e:                           # recorded, counted as 0 in B6_5
                        counts[run][c] = counts[run + "_countpy"][c] = 0
                        errors_run[c] = f"{type(e).__name__}: {str(e)[:200]}"
                if errors_run:
                    errors[run] = errors_run
        rec = {"key": it["key"], "image_path": str(it["image_path"]), "classes": it["query"],
               "classes_std": it.get("query_std"), "meta": it["meta"], "version": "OmniCount", "counts": counts,
               "diag": {"t_total_s": time.time() - t0, **ta, "per_class": diag_runs,
                        "errors": {f"{r}|{c}": m for r, em in errors.items() for c, m in em.items()}}}
        tmp = rows_dir / f"{it['key']}.json.{worker}.tmp"; tmp.write_text(json.dumps(rec)); tmp.replace(rows_dir / f"{it['key']}.json")
    return n_done

def run_shared(items, rows_dir, dataset, models, counter, refine_image, cache_dir=None, reverse=False, poll_s=120):
    """Run until every row exists: own pass, then re-pass after a pause (picks up rows the other session has not
    written yet — in flight, or left behind if it stopped). Returns images processed by THIS session."""
    rows_dir = Path(rows_dir); n_total = 0
    while True:
        n_total += run_omnicount(items, rows_dir, dataset, models, counter, refine_image, cache_dir, reverse)
        if all((rows_dir / f"{it['key']}.json").exists() for it in items):
            return n_total
        print(f"{dataset}: waiting for rows in flight in the other session ({time.strftime('%H:%M')})"); time.sleep(poll_s)

MODELS = {"marigold": marigold_pipe, "san": san_predictor, "seed_all": marigold_seeding.seed_all, "cfg": OMNI_CFG, "tqdm": tqdm,
          "worker": PART}
CACHE_ROOT = OUT_DIR / "cache"            # depth + SAN masks (lossless npz); FSC-147 only, so a gate review needs no new depth

CACHE_DATASETS = {"fsc147", "carpk", "voc", "omnicount"}   # 30 Oct: cache depth + SAN masks everywhere, so a re-run after an author reply reuses depth

def rows_of(name, report=False):
    """One row per image key. Two sessions writing the same image at the same moment (only near a meeting point) could
    leave two Drive entries for one key; both are valid (same image, same settings) and the first is kept."""
    rows, seen, dup = [], set(), 0
    for p in sorted((OUT_DIR / name / "rows_omnicount").glob("*.json")):
        r = json.load(open(p))
        if r["key"] in seen:
            dup += 1; continue
        seen.add(r["key"]); rows.append(r)
    if report and dup:
        print(f"{name}: {dup} duplicate Drive entries ignored (same key written by both sessions)")
    return rows

def single_metrics(rows, key):
    gt = np.array([r["meta"]["gt"][r["classes"][0]] for r in rows], float)
    pr = np.array([r["counts"][key][r["classes"][0]] for r in rows], float); e = pr - gt
    return {"images": len(rows), "MAE": np.abs(e).mean(), "RMSE": np.sqrt((e ** 2).mean()), "NAE": (np.abs(e) / gt).mean(),
            "SRE": np.sqrt(((e ** 2) / gt).mean()),
            "authors_NAE(sum|e|/sum y)": np.abs(e).sum() / gt.sum(), "authors_SRE(mean|e|/y)": (np.abs(e) / gt).mean(),
            "errors": sum(len(r["diag"]["errors"]) for r in rows)}

def diag_summary(rows, run="OmniCount"):
    d = [v for r in rows for v in r["diag"]["per_class"].get(run, {}).values()]
    if not d:
        return {}
    fb = pd.Series([x["fallback"] for x in d]).value_counts().to_dict()
    return {"queries": len(d), "fallback": fb, "san_empty": int(sum(x["san_empty"] for x in d)),
            "median_points": float(np.median([x["n_points"] for x in d]))}

# ---- Decisions fixed before any test count (written once; a later change stops the notebook unless re-declared on purpose)
DECLARATIONS = {
    "omnicount_commit": OMNI_COMMIT, "submodules": SUBMODULES,
    "per_dataset_settings": OMNI_CFG,
    "count_primary": "number of masks returned by the generator (incl. TFOC's small-region filter) -> counts['OmniCount'], counts['OmniCount_std']",
    "count_secondary": "visualization/count.py active rule: masks with overlay mean pixel value > class average -> counts['*_countpy']",
    "san": "ViT-L/14 (san_clip_vit_large_res4_coco.yaml); predict(img, [class], False) -> COCO-all merged; mask = argmax == class (reconstructed writer)",
    "marigold": {"checkpoint_order": ["Bingxin/Marigold", "prs-eth/marigold-depth-v1-0"], **MARIGOLD_ARGS, "seed_per_image": MARIGOLD_SEED, "dtype": "float32"},
    "refinement": {"tolerance": REFINE_TOL, "window": REFINE_WIN, "opening_radius": OPEN_RADIUS,
                   "mask_for_points": "extract_bin_mask.py (diamond)", "patch_for_sam": "extract_patch.py (disk)", "empty_mask": "depth < 0.5"},
    "other_masks": "the rest of the run's query set (present classes; or all classes on the fixed subsets)",
    "gate": {"published_fsc147_text_mae": 21.46, "relative_tolerance": 0.10},
}
DECL_PATH = OUT_DIR / "B6_4_declarations.json"
if DECL_PATH.exists():
    prev = json.load(open(DECL_PATH))
    assert json.loads(json.dumps(DECLARATIONS)) == prev["declarations"], \
        "Declarations differ from the saved file — stop; only change them deliberately (and record why)."
    print("Declarations unchanged since", prev["recorded_utc"])
else:
    json.dump({"recorded_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "declarations": DECLARATIONS},
              open(DECL_PATH, "w"), indent=2)
    print("Declarations recorded:", DECL_PATH)

Declarations unchanged since 2026-10-29T11:36:37+00:00


In [7]:
# ============================================================
# Step 1.1 — Sanity check before the long runs: 3 FSC-147 VAL images (no test image)
#   (a) port = authors' refinement functions (run verbatim from the cloned scripts) on REAL SAN masks + depth
#   (b) end-to-end counts, fallbacks, per-stage time -> runtime estimate for every dataset
# ============================================================
import ast, warnings
def _authors_fn(rel, name):
    src = (OMNI_DIR / rel).read_text(); tree = ast.parse(src)
    fn = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == name][-1]       # the active definition
    ns = {"np": np, "cv2": cv2}; exec("from scipy import ndimage\n" + ast.get_source_segment(src, fn), ns); return ns[name]
orig_bin = _authors_fn("preprocessing/extract_bin_mask.py", "process_binary_mask")
orig_patch = _authors_fn("preprocessing/extract_patch.py", "process_binary_mask")

fsc_val = D.FSC147Dataset(DATASETS / "FSC147" / "raw", split="val", load_images=False)
SANITY = []
for j, i in enumerate([0, 300, 600]):
    s = fsc_val[i]; c = next(iter(s["categories"]))
    extra = ["table", "floor"] if j == 1 else []                    # image 2: 3-class query so the "other masks" rule is exercised
    SANITY.append({"key": f"val{i:05d}", "image_path": s["image_path"], "query": [c] + [e for e in extra if e != c], "query_std": None,
                   "prompts": {k: k for k in [c] + extra}, "meta": {"gt": {c: s["categories"][c]["count"]}}})
n_checked, times = 0, []
for it in SANITY:
    t0 = time.time(); depth, san, ta = stage_a(it, it["query"], MODELS)
    rgb = cv2.cvtColor(cv2.imread(str(it["image_path"])), cv2.COLOR_BGR2RGB)
    dn = normalise_depth(depth)
    for c in it["query"]:                                            # (a) authors' loops vs port, class by class
        _, otsu = cv2.threshold(san[c], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        others = [san[k] for k in it["query"] if k != c]
        with warnings.catch_warnings(), np.errstate(all="ignore"), contextlib.redirect_stdout(io.StringIO()):
            warnings.simplefilter("ignore")
            a = orig_bin(otsu, others, dn, it["key"]); b = orig_patch(otsu, others, dn, it["key"])
            rb, pm, empty = refine_class(san[c], others, dn)
        assert np.array_equal(a, rb) and np.array_equal(np.asarray(b, bool), pm), f"refinement differs: {it['key']} {c}"
        n_checked += 1
    t1 = time.time(); ref = refine_image(san, depth, rgb); out = {}
    for c in it["query"]:
        n, n_cpy, d = counter.count(ref[c][0], ref[c][1], OMNI_CFG["fsc147"]); out[c] = (n, n_cpy, d["n_points"], d["fallback"])
    t_count = (time.time() - t1) / len(it["query"])
    times.append({"t_depth": ta["t_depth_s"], "t_san_per_class": ta["t_san_s"] / len(it["query"]), "t_count_per_class": t_count})
    c0 = it["query"][0]
    print(f"{it['key']}: GT {it['meta']['gt'][c0]} | " + " | ".join(f"{k}: masks {v[0]}, countpy {v[1]}, points {v[2]}, fallback {v[3]}" for k, v in out.items())
          + f" | depth {ta['t_depth_s']:.1f}s, SAN {ta['t_san_s']:.1f}s, count {t_count:.1f}s/class")
print(f"REFINEMENT EQUIVALENCE: PASS ({n_checked} class masks on real images identical to the authors' functions)")

tm = pd.DataFrame(times).mean()
est = {}
for k, v in ITEMS.items():
    n_img = len(v); n_san = sum(len(set(it["query"]) | set(it["query_std"] or [])) for it in v)
    n_cnt = sum(len(it["query"]) + len(it["query_std"] or []) for it in v)
    est[k] = (n_img * tm["t_depth"] + n_san * tm["t_san_per_class"] + n_cnt * tm["t_count_per_class"]) / 3600
print("Estimated hours (L4, from these 3 images; dense images take longer to count):",
      {k: round(float(h), 1) for k, h in est.items()}, "| total", round(float(sum(est.values())), 1))

/content/OmniCount/external/SAN/predict.py:227: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  image = torch.from_numpy(np.asarray(image)).float()


val00000: GT 13 | seagulls: masks 6, countpy 1, points 15, fallback none | depth 51.7s, SAN 5.2s, count 1.8s/class
val00300: GT 117 | bottle caps: masks 108, countpy 48, points 191, fallback none | table: masks 71, countpy 37, points 159, fallback none | floor: masks 71, countpy 37, points 159, fallback none | depth 33.3s, SAN 0.8s, count 7.1s/class
val00600: GT 14 | sausages: masks 15, countpy 8, points 91, fallback none | depth 30.0s, SAN 0.3s, count 4.9s/class
REFINEMENT EQUIVALENCE: PASS (5 class masks on real images identical to the authors' functions)
Estimated hours (L4, from these 3 images; dense images take longer to count): {'fsc147': 14.8, 'carpk': 5.7, 'voc': 84.5, 'omnicount': 31.4} | total 136.5


In [8]:
# ============================================================
# Step 2.1 — OmniCount on FSC-147 test (both sessions share it: A from the start, B from the end) + REPRODUCTION GATE
#            (published text MAE 21.46; both sessions stop if outside ±10%)
# ============================================================
GATE_PUBLISHED, GATE_REL_TOLERANCE = 21.46, 0.10
OVERRIDE_GATE = True          # set True only after reviewing the gap (it is then documented, and the other datasets run) - reviewed in B6_4v_OmniCount_FSC_grid64_A(or B).ipynb
t0 = time.time()
n = run_shared(ITEMS["fsc147"], OUT_DIR / "fsc147" / "rows_omnicount", "fsc147", MODELS, counter, refine_image,
               cache_dir=CACHE_ROOT / "fsc147" if "fsc147" in CACHE_DATASETS else None, reverse=(PART == "B"))
print(f"fsc147: processed now by {PART}: {n} | {(time.time() - t0) / 3600:.2f} h")
rows = rows_of("fsc147"); assert len(rows) == len(ITEMS["fsc147"]), f"{len(rows)}/{len(ITEMS['fsc147'])} rows — Run all again"
m, m_cpy = single_metrics(rows, "OmniCount"), single_metrics(rows, "OmniCount_countpy")
print("OmniCount FSC-147 test (primary: number of masks):", {k: round(float(v), 3) for k, v in m.items()})
print("OmniCount FSC-147 test (secondary: count.py rule): ", {k: round(float(v), 3) for k, v in m_cpy.items()})
print("Published (OmniCount Table 2, text): MAE 21.46 | RMSE 133.28 | 'NAE' 0.32 | 'SRE' 0.39 (authors' definitions, see header)")
print("Diagnostics:", diag_summary(rows))
lo, hi = GATE_PUBLISHED * (1 - GATE_REL_TOLERANCE), GATE_PUBLISHED * (1 + GATE_REL_TOLERANCE)
REPRO = {"omnicount_fsc147_primary": m, "omnicount_fsc147_countpy": m_cpy, "diagnostics": diag_summary(rows),
         "published": {"MAE": 21.46, "RMSE": 133.28, "NAE_authors": 0.32, "SRE_authors": 0.39},
         "gate": {"range": [lo, hi], "pass": bool(lo <= m["MAE"] <= hi), "override": OVERRIDE_GATE}}
if PART == "A":                                                    # one writer for the shared file
    json.dump(REPRO, open(OUT_DIR / "B6_4_omnicount_reproduction.json", "w"), indent=2, default=float)
if not lo <= m["MAE"] <= hi and not OVERRIDE_GATE:
    raise SystemExit(f"REPRODUCTION GATE: MAE {m['MAE']:.2f} is outside {lo:.2f}–{hi:.2f} (21.46 ± 10%) — stop and review before the other datasets.")
print("REPRODUCTION GATE:", "PASS" if lo <= m["MAE"] <= hi else "OVERRIDDEN after review")

OmniCount fsc147 [A]: 0it [00:00, ?it/s]

fsc147: processed now by A: 0 | 0.00 h
OmniCount FSC-147 test (primary: number of masks): {'images': 1190.0, 'MAE': 29.339, 'RMSE': 137.526, 'NAE': 0.452, 'SRE': 4.893, 'authors_NAE(sum|e|/sum y)': 0.443, 'authors_SRE(mean|e|/y)': 0.452, 'errors': 0.0}
OmniCount FSC-147 test (secondary: count.py rule):  {'images': 1190.0, 'MAE': 44.625, 'RMSE': 148.223, 'NAE': 0.554, 'SRE': 5.824, 'authors_NAE(sum|e|/sum y)': 0.674, 'authors_SRE(mean|e|/y)': 0.554, 'errors': 0.0}
Published (OmniCount Table 2, text): MAE 21.46 | RMSE 133.28 | 'NAE' 0.32 | 'SRE' 0.39 (authors' definitions, see header)
Diagnostics: {'queries': 1190, 'fallback': {'none': 1136, 'few_masks': 38, 'few_points': 16}, 'san_empty': 58, 'median_points': 99.0}
REPRODUCTION GATE: OVERRIDDEN after review


In [9]:
# ============================================================
# Step 3.1 — CARPK, VOC 2007, OmniCount-191 shared by the two sessions (resumable)
#   A: VOC -> OmniCount-191 -> CARPK (from the start of each list); B: CARPK -> OmniCount-191 -> VOC (from the end)
# ============================================================
RUN = ["voc", "omnicount", "carpk"] if PART == "A" else ["carpk", "omnicount", "voc"]
for name in RUN:
    t0 = time.time()
    n = run_shared(ITEMS[name], OUT_DIR / name / "rows_omnicount", name, MODELS, counter, refine_image,
                   cache_dir=CACHE_ROOT / name if name in CACHE_DATASETS else None, reverse=(PART == "B"))
    print(f"{name}: processed now by {PART}: {n} | {(time.time() - t0) / 3600:.2f} h")

OmniCount voc [A]:   0%|          | 0/1007 [00:00<?, ?it/s]

voc: processed now by A: 536 | 8.15 h


OmniCount omnicount [A]: 0it [00:00, ?it/s]

omnicount: processed now by A: 0 | 0.00 h


OmniCount carpk [A]: 0it [00:00, ?it/s]

carpk: processed now by A: 0 | 0.00 h


In [10]:
# ============================================================
# Step 4.1 — Completeness, progress summary (descriptive; B6_5 runs the pre-declared analysis), metadata
# ============================================================
import src.evaluation as ev
status, summary = {}, []
for name in ["fsc147", "carpk", "voc", "omnicount"]:
    d = OUT_DIR / name / "rows_omnicount"
    done = len(rows_of(name, report=True)) if d.exists() else 0           # unique image keys
    status[f"OmniCount|{name}"] = f"{done}/{len(ITEMS[name])}"
    if done < len(ITEMS[name]):
        continue
    rows = rows_of(name)
    if name in ("fsc147", "carpk"):
        for key in ["OmniCount", "OmniCount_countpy"]:
            summary.append({"dataset": name, "method": key, "protocol": "single-class", **single_metrics(rows, key)})
        continue
    gk = "gt_all" if name == "voc" else "gt"
    for key in ["OmniCount", "OmniCount_countpy"]:
        present = [{"image_id": r["meta"]["image_id"], "domain": r["meta"]["domain"],
                    "categories": {c: {"count": n} for c, n in r["meta"][gk].items()},
                    "pred_counts": r["counts"][key], "query_classes": list(r["meta"][gk])} for r in rows]
        sub = [{"image_id": r["meta"]["image_id"], "domain": r["meta"]["domain"],
                "categories": {c: {"count": n} for c, n in r["meta"][gk].items()},
                "pred_counts": r["counts"][key.replace("OmniCount", "OmniCount_std")], "query_classes": r["classes_std"]}
               for r in rows if r["meta"]["in_standard_subset"]]
        for proto, res in [("present classes (full test)", present), ("standard (subset)", sub)]:
            _, _, m = ev.evaluate_multi_label_counts(ev.build_multi_label_pairs(res))
            summary.append({"dataset": name, "method": key, "protocol": proto, "images": len(res), "MAE_nz": m["MAE_nz"],
                            "mRMSE-nz": m["mRMSE_nz"], "standard mRMSE": m["mRMSE"] if proto.startswith("standard") else np.nan,
                            "FP units": m["false_positive_units"], "errors": sum(len(r["diag"]["errors"]) for r in rows)})
print("Status:", status)
if summary:
    sm = pd.DataFrame(summary); display(sm.round(3)); sm.to_csv(OUT_DIR / f"B6_4_progress_summary_{PART}.csv", index=False)
complete = all(v.split("/")[0] == v.split("/")[1] for v in status.values())
json.dump({"notebook": f"B6_4_Baseline_OmniCount_{PART}", "part": PART, "omnicount_commit": OMNI_HEAD, "submodules": SUB_HEADS, "repo_commit": REPO_COMMIT,
           "gpu": GPU_NAME, "marigold_checkpoint": MARIGOLD_CKPT, "san_checkpoint_sha256": SAN_L_SHA256,
           "patches": [f"{f}: {o} -> {n.split('  #')[0]}" for f, o, n in PATCHES],
           "reconstructed": "SAN binary-mask writer (predict.py, one class per call, argmax == class)",
           "engineering_changes": ["models loaded once", "unused CLIPSeg call removed", "refinement vectorised (verified identical)",
                                   "fixed-list CLIP text features computed once", "Marigold seed 0 per image (wrapper: clock)",
                                   "Marigold colour map skipped (output unused)"],
           "declarations_file": str(DECL_PATH), "rows_counts_keys": ["OmniCount", "OmniCount_countpy", "OmniCount_std", "OmniCount_std_countpy"],
           "diag_errors_key_format": "run|class", "status": status, "complete": complete,
           "rule_file_recorded_utc": RULE["recorded_utc"], "torch": torch.__version__,
           "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}, open(OUT_DIR / f"B6_4_metadata_{PART}.json", "w"), indent=2)
print("COMPLETE" if complete else "NOT COMPLETE — Run all again to continue", "| saved to", OUT_DIR)

fsc147: 1 duplicate Drive entries ignored (same key written by both sessions)
voc: 1 duplicate Drive entries ignored (same key written by both sessions)
Status: {'OmniCount|fsc147': '1190/1190', 'OmniCount|carpk': '459/459', 'OmniCount|voc': '4952/4952', 'OmniCount|omnicount': '1957/1957'}


,dataset,method,protocol,images,MAE,RMSE,NAE,SRE,authors_NAE(sum|e|/sum y),authors_SRE(mean|e|/y),errors,MAE_nz,mRMSE-nz,standard mRMSE,FP units
0,fsc147,OmniCount,single-class,1190,29.339,137.526,0.452,4.893,0.443,0.452,0,NaN,NaN,NaN,NaN
1,fsc147,OmniCount_countpy,single-class,1190,44.625,148.223,0.554,5.824,0.674,0.554,0,NaN,NaN,NaN,NaN
2,carpk,OmniCount,single-class,459,49.673,55.140,0.535,5.218,0.480,0.535,0,NaN,NaN,NaN,NaN
3,carpk,OmniCount_countpy,single-class,459,77.741,83.110,0.756,7.681,0.751,0.756,0,NaN,NaN,NaN,NaN
4,voc,OmniCount,present classes (full test),4952,NaN,NaN,NaN,NaN,NaN,NaN,0,8.717,15.542,NaN,0.0
5,voc,OmniCount,standard (subset),500,NaN,NaN,NaN,NaN,NaN,NaN,0,7.958,12.979,31.324,218008.0
6,voc,OmniCount_countpy,present classes (full test),4952,NaN,NaN,NaN,NaN,NaN,NaN,0,2.042,4.006,NaN,0.0
7,voc,OmniCount_countpy,standard (subset),500,NaN,NaN,NaN,NaN,NaN,NaN,0,1.819,3.190,7.687,50738.0
8,omnicount,OmniCount,present classes (full test),1957,NaN,NaN,NaN,NaN,NaN,NaN,0,8.273,14.180,NaN,0.0
9,omnicount,OmniCount,standard (subset),303,NaN,NaN,NaN,NaN,NaN,NaN,0,4.036,5.120,6.587,8579.0


COMPLETE | saved to /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B6_4_Baseline_OmniCount
